In [14]:
import torch
import torch.nn as nn

In [15]:
from GPT_Model import GPT,generate_text,cfg,tokenizer,text_to_token_id,token_id_to_text

In [ ]:
torch.manual_seed(123)
model=GPT(cfg)
model.eval()


GPT(
  (token_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (tranformer_block): Sequential(
    (0): Transfomers(
      (layernorm): LayerNorm()
      (mutihead_atten): MultiHeadAttention(
        (w_query): Linear(in_features=768, out_features=768, bias=False)
        (w_key): Linear(in_features=768, out_features=768, bias=False)
        (w_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (feed_forward): FeedForward(
        (layer): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
    )
    (1): Transfomers(
      (layernorm): LayerNorm()
      (mutihead_atten): MultiHeadAttention(
        (w_

## Model Evaluation

In [17]:
start_context = "Every effort moves you"
token_id=generate_text(
  model=model,
 ip_token_id=text_to_token_id(start_context,tokenizer),
  max_new_tokens=5,
  context_size=cfg["context_len"]
)
text=(token_id_to_text(token_id,tokenizer))

In [18]:
inputs = torch.tensor([
    [16833, 3626, 6100], # Every effort moves
    [40, 1107, 588]      # I really like
])

targets = torch.tensor([
    [3626, 6100, 345],   # effort moves you
    [1107, 588, 11311]   # really like chocolate
])

<div style="text-align: center;">
 <img width=700px heigth=20px src="./IMG/loss_Train.png" alt="Example image">
 </div>

<div style="text-align: center;">
 <img width=700px heigth=20px src="./IMG/loss_cal.png" alt="Example image">
 </div>

In [ ]:
#find Logits,Probabilty
with torch.no_grad():
  logit=model(inputs)
prob=torch.softmax(logit,dim=-1)  #converted to prob
print(prob.shape)

torch.Size([2, 3, 50257])


In [ ]:
#fancy indexing
X = np.arange(12).reshape((3, 4))
print(X)
row = np.array([0, 1, 2])
col = np.array([2, 1, 3])
X[row, col]  #in particular row we have to find col value

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]


array([ 2,  5, 11])

In [29]:
#find the target_probabilty from predicted probabilty and want to max that value
target_probs=[]
target_probs.append(prob[0 , [0,1,2], targets[0]])
print(f"Text {0+1}: {target_probs[0]}")

target_probs.append(prob[1 , [0,1,2], targets[1]])
print(f"Text {0+1}: {target_probs[0]}")

Text 1: tensor([3.0695e-05, 1.8850e-05, 1.3407e-05])
Text 1: tensor([3.0695e-05, 1.8850e-05, 1.3407e-05])


In [ ]:
#concat all batch probabilty then take log  
log_prob=torch.log(torch.cat((target_probs[0],target_probs[1])))

In [32]:
#take avg
avg_log_prob=torch.mean(log_prob)

In [33]:
#take neg
neg_avg_log_prob=-1*avg_log_prob
print(neg_avg_log_prob)

tensor(10.8026)


### Doing same thing as above using predefined cross entropy loss in torch

In [34]:
loss=torch.nn.functional.cross_entropy(logit.flatten(0,1),targets.flatten())
print(loss)

tensor(10.8026)


## Perplexity

<div style="text-align: center;">
 <img width=700px heigth=20px src="./IMG/perplexity.png" alt="Example image">
 </div>

In [35]:
perplexity=torch.exp(loss)
print(perplexity)

tensor(49150.1992)
